In [ ]:
# EDA — HackYeah sessions

W tym notatniku przeprowadzimy eksploracyjną analizę danych z sesji z katalogu `data/sessions`.

Celem jest szybkie rozpoznanie struktury zbioru, jakości danych, rozkładów kluczowych cech oraz przygotowanie danych do dalszej analizy lub modelowania.

## Źródło danych

- katalog sesji: `/workspace/hackaton/HackYeach-2026-REPO/data/sessions`
- struktura sesji: `meta.json`, `events.jsonl`, `face.jsonl`, `net.jsonl`, `video/*`
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

ROOT = Path("/workspace/hackaton/HackYeach-2026-REPO")
DATA_DIR = ROOT / "data" / "sessions"

print(f"ROOT: {ROOT}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"Czy katalog istnieje: {DATA_DIR.exists()}")
print(f"Liczba podkatalogów: {len(list(DATA_DIR.iterdir())) if DATA_DIR.exists() else 0}")

session_dirs = sorted([p for p in DATA_DIR.iterdir() if p.is_dir()])
print(f"Liczba sesji: {len(session_dirs)}")
print("Pierwsze 5 sesji:")
for p in session_dirs[:5]:
    print(" -", p.name)
# Wczytanie danych z sesji

rows = []
for session_dir in session_dirs:
    meta_path = session_dir / "meta.json"
    if not meta_path.exists():
        continue
    try:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
    except Exception:
        meta = {}

    events_path = session_dir / "events.jsonl"
    face_path = session_dir / "face.jsonl"
    net_path = session_dir / "net.jsonl"
    video_dir = session_dir / "video"

    row = {
        "session_id": meta.get("session_id") or session_dir.name,
        "participant_code": meta.get("participant_code"),
        "condition": meta.get("condition"),
        "order_index": meta.get("order_index"),
        "started_at": meta.get("started_at"),
        "ended_at": meta.get("ended_at"),
        "has_events": events_path.exists(),
        "has_face": face_path.exists(),
        "has_net": net_path.exists(),
        "video_files": len(list(video_dir.glob("*"))) if video_dir.exists() else 0,
        "survey_present": (session_dir / "survey.json").exists(),
    }
    rows.append(row)

sessions_df = pd.DataFrame(rows)
print(sessions_df.head())
print("\nTypy kolumn:")
print(sessions_df.dtypes)

if sessions_df.empty:
    raise ValueError("Brak sesji do analizy.")
# Przegląd struktury zbioru

print(f"Liczba wierszy: {sessions_df.shape[0]}")
print(f"Liczba kolumn: {sessions_df.shape[1]}")
print("\nNazwy kolumn:")
print(list(sessions_df.columns))
print("\nPodstawowe informacje:")
sessions_df.info()

sessions_df.describe(include=["number"]).T
# Ocena braków danych i duplikatów

missing = sessions_df.isna().sum().sort_values(ascending=False)
print("Braki danych:")
print(missing[missing > 0])

print("\nDuplikaty:")
print(f"Liczba zduplikowanych rekordów: {sessions_df.duplicated().sum()}")

# wykres braków danych
plt.figure(figsize=(10, 5))
missing.plot(kind="bar", color="steelblue")
plt.title("Braki danych w kolumnach")
plt.ylabel("Liczba braków")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
# Statystyki opisowe

numeric_cols = [c for c in sessions_df.columns if pd.api.types.is_numeric_dtype(sessions_df[c])]
print("Kolumny numeryczne:")
print(numeric_cols)

if numeric_cols:
    print(sessions_df[numeric_cols].describe().T)

categorical_cols = [c for c in sessions_df.columns if c not in numeric_cols]
print("\nKolumny kategoryczne:")
print(categorical_cols)

if categorical_cols:
    print(sessions_df[categorical_cols].describe(include=["object", "category"]).T)
# Analiza rozkładów zmiennych numerycznych

# przygotowanie dodatkowych zmiennych pomocniczych
if "order_index" in sessions_df.columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(sessions_df["order_index"].dropna(), bins=20, kde=True)
    plt.title("Rozkład kolejności sesji")
    plt.xlabel("order_index")
    plt.tight_layout()
    plt.show()

if "video_files" in sessions_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=sessions_df["video_files"], color="lightblue")
    plt.title("Boxplot liczby plików video na sesję")
    plt.xlabel("liczba plików video")
    plt.tight_layout()
    plt.show()

if "started_at" in sessions_df.columns:
    started = pd.to_numeric(sessions_df["started_at"], errors="coerce").dropna()
    if not started.empty:
        plt.figure(figsize=(10, 5))
        sns.histplot(started, bins=30, kde=True)
        plt.title("Rozkład czasu rozpoczęcia sesji")
        plt.xlabel("started_at (timestamp)")
        plt.tight_layout()
        plt.show()
# Analiza zmiennych kategorycznych

if "condition" in sessions_df.columns:
    cond_counts = sessions_df["condition"].value_counts(dropna=False)
    print("Liczebności warunków:")
    print(cond_counts)

    plt.figure(figsize=(10, 5))
    sns.countplot(data=sessions_df, x="condition", order=cond_counts.index)
    plt.title("Liczba sesji wg warunku")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

if "participant_code" in sessions_df.columns:
    print("\nNajczęściej występujące kody uczestników:")
    print(sessions_df["participant_code"].value_counts().head())
# Analiza korelacji i zależności między zmiennymi

corr_df = sessions_df.select_dtypes(include=[np.number])
if not corr_df.empty:
    corr = corr_df.corr(numeric_only=True)
    print("Macierz korelacji:")
    print(corr)

    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Macierz korelacji zmiennych numerycznych")
    plt.tight_layout()
    plt.show()
else:
    print("Brak zmiennych numerycznych do analizy korelacji.")
# Wizualizacja trendów i wartości odstających

if {"order_index", "video_files"}.issubset(sessions_df.columns):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=sessions_df, x="order_index", y="video_files", hue="condition", s=80)
    plt.title("Liczba plików video w zależności od kolejności sesji")
    plt.xlabel("order_index")
    plt.ylabel("video_files")
    plt.tight_layout()
    plt.show()

if "condition" in sessions_df.columns and "video_files" in sessions_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=sessions_df, x="condition", y="video_files")
    plt.title("Rozkład liczby plików video wg warunku")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
# Przygotowanie danych do dalszej analizy lub modelowania

# Zastosowanie prostego, praktycznego planu przygotowania danych
# 1. usunięcie duplikatów
# 2. uzupełnienie braków w kolumnach istotnych
# 3. normalizacja typów danych
# 4. kodowanie kolumn kategorycznych
# 5. zabezpieczenie przed wartościami odstającymi

clean_df = sessions_df.copy()
clean_df = clean_df.drop_duplicates()

for col in ["order_index", "video_files"]:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

for col in ["condition", "participant_code"]:
    if col in clean_df.columns:
        clean_df[col] = clean_df[col].fillna("unknown")

print("Zbiór po przygotowaniu:")
print(clean_df.head())
print("\nLiczba rekordów po usunięciu duplikatów:", clean_df.shape[0])

# przykładowy podział na zakodowane kolumny kategoryczne
if "condition" in clean_df.columns:
    encoded = pd.get_dummies(clean_df[["condition"]], prefix=["condition"])
    print("\nPrzykład kodowania kategorii:")
    print(encoded.head())

print("\nKroki dalszego przygotowania danych:")
print("1. Uzupełnienie braków w kolumnach istotnych dla analizy.")
print("2. Kodowanie zmiennych kategorycznych (one-hot lub ordinal encoding).")
print("3. Normalizacja lub standaryzacja kolumn numerycznych przed modelowaniem.")
print("4. Weryfikacja wartości odstających i ich potencjalny wpływ na model.")
print("5. Rozważenie wyodrębnienia cech z `events.jsonl`, `face.jsonl` i `net.jsonl` do tabel per sesja.")